In [1]:
import sklearn
import xgboost
import joblib
import pandas as pd
import numpy as np

print(f"scikit-learn version : {sklearn.__version__}")
print(f"xgboost version      : {xgboost.__version__}")
print(f"joblib version       : {joblib.__version__}")
print(f"pandas version       : {pd.__version__}")

scikit-learn version : 1.6.1
xgboost version      : 3.4.1
joblib version       : 1.6.0
pandas version       : 2.2.3


In [3]:
import os
import json
import joblib
import numpy as np
import pandas as pd

# 1. Muat Dataset (Pastikan aug_train.csv sudah diunggah ke Google Colab)
DATASET_PATH = "aug_train.csv"

if not os.path.exists(DATASET_PATH):
    raise FileNotFoundError(
        f"File '{DATASET_PATH}' tidak ditemukan! Silakan unggah aug_train.csv dari Kaggle ke Google Colab."
    )

df_raw = pd.read_csv(DATASET_PATH)
print(f"Ukuran dataset awal: {df_raw.shape}")

# 2. Fungsi Konversi Kolom 'experience' dari String ('>20', '<1') menjadi Numerik
def parse_experience(val):
    if pd.isna(val):
        return np.nan
    val_str = str(val).strip()
    if val_str == ">20":
        return 21.0
    elif val_str == "<1":
        return 0.5
    try:
        return float(val_str)
    except ValueError:
        return np.nan

df = df_raw.copy()
df["experience_years"] = df["experience"].apply(parse_experience)

# 3. Penyesuaian Target & Sintesis Fitur Kualifikasi (IPK & Skor Tes)
# Catatan: Pada dataset asli Kaggle, target=1 berarti "ingin pindah kerja".
# Agar selaras dengan sistem seleksi kualifikasi kandidat:
# Kita bangun indeks kelayakan berbasis kualifikasi + target, serta menambahkan kolom GPA & Test Score
# jika belum tersedia di CSV.
np.random.seed(42)
n_samples = len(df)

if "gpa" not in df.columns or "test_score" not in df.columns:
    print("Menambahkan fitur kualifikasi terkalibrasi: 'gpa' (IPK) dan 'test_score' (Skor Tes)...")
    # Buat sinyal kualitas kandidat dari data asli (pengalaman, jam pelatihan, dan indeks kota)
    exp_norm = df["experience_years"].fillna(3.0) / 21.0
    train_norm = np.clip(df["training_hours"].fillna(40.0) / 150.0, 0, 1)
    cdi_norm = df["city_development_index"].fillna(0.75)
    has_rel_exp = (df["relevent_experience"] == "Has relevent experience").astype(float)

    quality_signal = (0.35 * exp_norm) + (0.25 * has_rel_exp) + (0.20 * train_norm) + (0.20 * cdi_norm)

    # Generate IPK (skala 2.20 - 4.00) dan Skor Tes Teknis (skala 35.0 - 100.0)
    df["gpa"] = np.clip(2.50 + (quality_signal * 1.35) + np.random.normal(0, 0.18, n_samples), 2.00, 4.00).round(2)
    df["test_score"] = np.clip(45.0 + (quality_signal * 48.0) + np.random.normal(0, 6.5, n_samples), 0.0, 100.0).round(1)

    # Kalibrasi target kelayakan (1 = Layak/Kualifikasi Sesuai, 0 = Belum Memenuhi Kualifikasi)
    # berdasarkan kombinasi kualifikasi objektif agar model belajar pola penerimaan yang logis
    suitability_score = (
        0.30 * ((df["gpa"] - 2.0) / 2.0)
        + 0.35 * (df["test_score"] / 100.0)
        + 0.20 * exp_norm
        + 0.15 * has_rel_exp
    )
    noise = np.random.normal(0, 0.05, n_samples)
    df["is_qualified"] = ((suitability_score + noise) >= 0.58).astype(int)
else:
    df["is_qualified"] = df["target"].astype(int)

# 4. Pilih Fitur Utama sesuai Rumusan Masalah
NUMERIC_FEATURES = [
    "gpa",                      # IPK Kandidat (2.00 - 4.00)
    "test_score",               # Skor Tes Teknis (0 - 100)
    "experience_years",         # Lama Pengalaman Kerja (Tahun)
    "training_hours",           # Jam Pelatihan/Sertifikasi
    "city_development_index"    # Indeks Perkembangan Kota/Domisili (0.4 - 1.0)
]

CATEGORICAL_FEATURES = [
    "education_level",          # Graduate (S1), Masters (S2), Phd (S3), High School, Primary School
    "major_discipline",         # STEM, Business Degree, Arts, Humanities, No Major, Other
    "relevent_experience",      # Has relevent experience / No relevent experience
    "enrolled_university"       # no_enrollment, Full time course, Part time course
]

ALL_FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES
TARGET_COL = "is_qualified"

df_clean = df[ALL_FEATURES + [TARGET_COL]].copy()
print("\nDistribusi Target Kelayakan (0 = Belum Layak, 1 = Layak):")
print(df_clean[TARGET_COL].value_counts(normalize=True).round(3))
df_clean.head()

Ukuran dataset awal: (19158, 14)
Menambahkan fitur kualifikasi terkalibrasi: 'gpa' (IPK) dan 'test_score' (Skor Tes)...

Distribusi Target Kelayakan (0 = Belum Layak, 1 = Layak):
is_qualified
1    0.69
0    0.31
Name: proportion, dtype: float64


,gpa,test_score,experience_years,training_hours,city_development_index,education_level,major_discipline,relevent_experience,enrolled_university,is_qualified
0,3.71,74.9,21.0,36,0.920,Graduate,STEM,Has relevent experience,no_enrollment,1
1,3.11,59.6,15.0,47,0.776,Graduate,STEM,No relevent experience,no_enrollment,0
2,3.05,54.9,5.0,83,0.624,Graduate,STEM,No relevent experience,Full time course,0
3,3.09,58.4,0.5,52,0.789,Graduate,Business Degree,No relevent experience,NaN,0
4,3.49,83.4,21.0,8,0.767,Masters,STEM,Has relevent experience,no_enrollment,1


In [4]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, roc_auc_score, f1_score

# 1. Split Data (80% Training, 20% Testing)
X = df_clean[ALL_FEATURES]
y = df_clean[TARGET_COL]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# 2. Bangun Preprocessor Pipeline
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, NUMERIC_FEATURES),
        ("cat", categorical_transformer, CATEGORICAL_FEATURES)
    ]
)

# Fit & Transform pada data latih, Transform pada data uji
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

# 3. Latih Kandidat Model 1: Random Forest Classifier
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    min_samples_split=5,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train_processed, y_train)
rf_probs = rf_model.predict_proba(X_test_processed)[:, 1]
rf_preds = (rf_probs >= 0.5).astype(int)
rf_auc = roc_auc_score(y_test, rf_probs)
rf_f1 = f1_score(y_test, rf_preds)

# 4. Latih Kandidat Model 2: XGBoost Classifier
xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric="logloss"
)
xgb_model.fit(X_train_processed, y_train)
xgb_probs = xgb_model.predict_proba(X_test_processed)[:, 1]
xgb_preds = (xgb_probs >= 0.5).astype(int)
xgb_auc = roc_auc_score(y_test, xgb_probs)
xgb_f1 = f1_score(y_test, xgb_preds)

print("=== PERBANDINGAN PERFORMA MODEL ===")
print(f"Random Forest -> F1-Score: {rf_f1:.4f} | ROC-AUC: {rf_auc:.4f}")
print(f"XGBoost       -> F1-Score: {xgb_f1:.4f} | ROC-AUC: {xgb_auc:.4f}")

# Pilih model terbaik secara otomatis berdasarkan ROC-AUC
if xgb_auc >= rf_auc:
    best_model = xgb_model
    best_model_name = "XGBoostClassifier"
    best_probs = xgb_probs
    best_preds = xgb_preds
else:
    best_model = rf_model
    best_model_name = "RandomForestClassifier"
    best_probs = rf_probs
    best_preds = rf_preds

print(f"\nModel Terbaik Terpilih: {best_model_name}")
print("\nClassification Report Model Terbaik:")
print(classification_report(y_test, best_preds, target_names=["Belum Layak (0)", "Layak (1)"]))

=== PERBANDINGAN PERFORMA MODEL ===
Random Forest -> F1-Score: 0.9518 | ROC-AUC: 0.9845
XGBoost       -> F1-Score: 0.9576 | ROC-AUC: 0.9844

Model Terbaik Terpilih: RandomForestClassifier

Classification Report Model Terbaik:
                 precision    recall  f1-score   support

Belum Layak (0)       0.89      0.90      0.89      1189
      Layak (1)       0.95      0.95      0.95      2643

       accuracy                           0.93      3832
      macro avg       0.92      0.92      0.92      3832
   weighted avg       0.93      0.93      0.93      3832



In [5]:
os.makedirs("exported_models", exist_ok=True)

# 1. Simpan Classifier Model & Preprocessor Pipeline
MODEL_FILE = "exported_models/classifier_model.joblib"
PREPROCESSOR_FILE = "exported_models/preprocessor_pipeline.joblib"
METADATA_FILE = "exported_models/feature_metadata.json"

joblib.dump(best_model, MODEL_FILE)
joblib.dump(preprocessor, PREPROCESSOR_FILE)

# 2. Hitung Nilai Benchmark (Nilai Ideal dari Kandidat yang Lolos) untuk Gap Analyzer
qualified_df = df_clean[df_clean[TARGET_COL] == 1]

ideal_benchmarks = {
    "gpa": round(float(qualified_df["gpa"].median()), 2),
    "test_score": round(float(qualified_df["test_score"].median()), 1),
    "experience_years": round(float(qualified_df["experience_years"].median()), 1),
    "training_hours": round(float(qualified_df["training_hours"].median()), 0),
    "city_development_index": round(float(qualified_df["city_development_index"].median()), 2)
}

# 3. Susun Metadata Lengkap untuk Aplikasi Web Lokal
feature_metadata = {
    "model_name": best_model_name,
    "sklearn_version": sklearn.__version__,
    "numeric_features": NUMERIC_FEATURES,
    "categorical_features": CATEGORICAL_FEATURES,
    "all_features_order": ALL_FEATURES,
    "status_thresholds": {
        "terima_min_prob": 0.75,
        "pertimbangan_min_prob": 0.45
    },
    "ideal_numeric_benchmarks": ideal_benchmarks,
    "categorical_options": {
        "education_level": ["Graduate", "Masters", "Phd", "High School", "Primary School"],
        "major_discipline": ["STEM", "Business Degree", "Arts", "Humanities", "No Major", "Other"],
        "relevent_experience": ["Has relevent experience", "No relevent experience"],
        "enrolled_university": ["no_enrollment", "Full time course", "Part time course"]
    }
}

with open(METADATA_FILE, "w", encoding="utf-8") as f:
    json.dump(feature_metadata, f, indent=4)

print("Ketiga artefak berhasil diekspor ke folder 'exported_models/':")
print(f"1. {MODEL_FILE}")
print(f"2. {PREPROCESSOR_FILE}")
print(f"3. {METADATA_FILE}")
print("\nIsi Benchmark Kualifikasi Ideal:")
print(json.dumps(ideal_benchmarks, indent=2))

Ketiga artefak berhasil diekspor ke folder 'exported_models/':
1. exported_models/classifier_model.joblib
2. exported_models/preprocessor_pipeline.joblib
3. exported_models/feature_metadata.json

Isi Benchmark Kualifikasi Ideal:
{
  "gpa": 3.45,
  "test_score": 78.5,
  "experience_years": 12.0,
  "training_hours": 50.0,
  "city_development_index": 0.91
}


In [6]:
from google.colab import files

# 1. Muat ulang artefak dari disk (Simulasi cara kerja aplikasi web lokal)
loaded_preprocessor = joblib.load(PREPROCESSOR_FILE)
loaded_model = joblib.load(MODEL_FILE)
with open(METADATA_FILE, "r") as f:
    loaded_meta = json.load(f)

def map_status(prob: float, thresholds: dict) -> str:
    if prob >= thresholds["terima_min_prob"]:
        return "Terima"
    elif prob >= thresholds["pertimbangan_min_prob"]:
        return "Pertimbangan"
    return "Tolak"

# 2. Buat 3 sampel kandidat uji (Kuat, Menengah, Lemah)
test_applicants = pd.DataFrame([
    {
        # Kandidat 1: Kualifikasi Kuat
        "gpa": 3.75, "test_score": 88.0, "experience_years": 6.0, "training_hours": 95.0,
        "city_development_index": 0.92, "education_level": "Masters", "major_discipline": "STEM",
        "relevent_experience": "Has relevent experience", "enrolled_university": "no_enrollment"
    },
    {
        # Kandidat 2: Kualifikasi Menengah (Borderline)
        "gpa": 3.15, "test_score": 68.0, "experience_years": 2.0, "training_hours": 40.0,
        "city_development_index": 0.78, "education_level": "Graduate", "major_discipline": "STEM",
        "relevent_experience": "Has relevent experience", "enrolled_university": "Part time course"
    },
    {
        # Kandidat 3: Kualifikasi Kurang
        "gpa": 2.45, "test_score": 48.0, "experience_years": 0.5, "training_hours": 12.0,
        "city_development_index": 0.62, "education_level": "High School", "major_discipline": "No Major",
        "relevent_experience": "No relevent experience", "enrolled_university": "Full time course"
    }
])

# Pastikan urutan kolom sesuai metadata
test_applicants = test_applicants[loaded_meta["all_features_order"]]
X_sim = loaded_preprocessor.transform(test_applicants)
sim_probs = loaded_model.predict_proba(X_sim)[:, 1]

print("=== HASIL UJI SIMULASI 3 PROFIL KANDIDAT ===")
for idx, prob in enumerate(sim_probs, start=1):
    status = map_status(prob, loaded_meta["status_thresholds"])
    print(f"Kandidat #{idx} -> Probabilitas: {prob*100:.2f}% | Status Prediksi: [{status}]")

# 3. Unduh otomatis ketiga file ke komputer lokal Anda
print("\nMengunduh ketiga artefak model ke komputer Anda...")
files.download(MODEL_FILE)
files.download(PREPROCESSOR_FILE)
files.download(METADATA_FILE)

=== HASIL UJI SIMULASI 3 PROFIL KANDIDAT ===
Kandidat #1 -> Probabilitas: 97.42% | Status Prediksi: [Terima]
Kandidat #2 -> Probabilitas: 49.28% | Status Prediksi: [Pertimbangan]
Kandidat #3 -> Probabilitas: 0.01% | Status Prediksi: [Tolak]

Mengunduh ketiga artefak model ke komputer Anda...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>